### Incluir los nombres de los colectores en la tabla Persons

Dado que los nombres de colectores fueron insertados en la tabla Occurrences como texto, es necesario generar las entradas homologadas en la tabla Persons y asociar dichas entradas a la tabla Occurrences usando el PersonID. En este sentido, este Notebook es de único uso.

Las homologaciones de las grafías redundantes de nombres de colectores fueron realizadas en LibreOffice Calc y se encuentran en el archivo `collectors.csv`.

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
from datetime import datetime

import credentials

In [2]:
conn_str = 'mysql+mysqlconnector://' + \
	f"{credentials.mysql['username']}:{credentials.mysql['password']}" + \
	'@localhost:3306/Mutis'
engine = create_engine(conn_str)
connection = engine.connect()

In [3]:
colls = pd.read_csv("collectors.csv", delimiter="\t")
colls["Standarized_Collector"] = colls.Standarized_Collector.str.replace(".", ". "
	).str.replace(",", ", "
	).str.replace(r"\.\s+,", ".,", regex=True
	).str.replace(r"\s+", " ", regex=True
	).str.replace(r"\s+$|^\s+", "", regex=True
	)

In [4]:
persons = pd.read_sql_table("Persons", connection)

# 1

Si no se han ingresado ningún colector de la tabla a la base de datos, ejecute las siguiente celdas, de lo contrario salte a 2

In [54]:
uniqcolls = colls[
	~colls.Standarized_Collector.isin(persons.NameVerbatim)
	].groupby("Standarized_Collector"
	).size(
	).reset_index(
	).drop(columns=0)

startidx = persons.PersonID.max() + 1
uniqcolls["PersonID"] = uniqcolls.index + startidx
uniqcolls[["LastName", "FirstName"]] = uniqcolls.Standarized_Collector.str.split(
	r",\s+", regex=True, expand=True
	)
uniqcolls[["Abbreviation", "Affiliation", "Email"]] = None, None, None
uniqcolls = uniqcolls.rename(columns={"Standarized_Collector":"NameVerbatim"})
uniqcolls["TimeStamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# Check no duplicates with records already in db

for row in uniqcolls.itertuples():
	q = persons.loc[persons.LastName == row.LastName]
	if q.shape[0] > 0:
		print(row.NameVerbatim)
		print(persons.loc[persons.LastName == row.LastName, "NameVerbatim"].values)
		print("=" * 40)

Barbosa, O.
['Barbosa, C.']
Betancur, A.
['Betancur, Julio']
Castellanos, J. D.
['Castellanos, C.']
Cuellar, Y.
['Cuellar, D. M.']
Daniel, R.
['Daniel, H.']
Escobar, W.
['Escobar, L. K.']
Gamboa, C.
['Gamboa, M. A.']
Gamboa, M. C.
['Gamboa, M. A.']
García, A.
['García, Y.']
García, B.
['García, Y.']
García, C.
['García, Y.']
García, F.
['García, Y.']
García, G.
['García, Y.']
García, H.
['García, Y.']
García, L. A. de
['García, Y.']
García, M.
['García, Y.']
García, M. C.
['García, Y.']
García, N.
['García, Y.']
García, Ni.
['García, Y.']
García, R.
['García, Y.']
García, Roger Fabián
['García, Y.']
García, Yamile
['García, Y.']
Gil, C. E.
['Gil, E.']
Hernández, G.
['Hernández, A.']
Hernández, J.
['Hernández, A.']
Hernández, L.
['Hernández, A.']
Hernández, M.
['Hernández, A.']
Hernández, N.
['Hernández, A.']
Hernández, Nicolás
['Hernández, A.']
Lewis, W. H.
['Lewis, G. P.']
Mayorga, C.
['Mayorga, M.' 'Mayorga, N. C.']
Montenegro, A.
['Montenegro, P. A.']
Montenegro, E.
['Montenegro, P.

In [57]:
uniqcolls[persons.columns].to_sql('Persons', engine, if_exists='append', index=False, 
		chunksize=10000, method='multi'
	)

1193

# 2

Se obtienen los PersonID de la BD Mutis que corresponden a todos los colectores verbatim de la tabla ocurrencias.

In [6]:
persons = pd.read_sql_table("Persons", connection)

In [13]:
colls.shape

(1419, 4)

In [ ]:
# Check after this join that there aren't missing PersonIDs in the original set of collector names

colls = colls.merge(
	persons[["NameVerbatim", "PersonID"]],
	left_on="Standarized_Collector",
	right_on="NameVerbatim",
	how="left"
)

In [6]:
peoper = pd.read_sql_table("PeoplePersons", connection)

In [ ]:
people = pd.read_sql_table("People", connection)
peoplepoint = people.PeopleID.max()
peoplepoint += 1
thnow = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

for acoll in colls.PersonID.unique():
	#print(acoll)
	unitary = False
	s = text("SELECT People, Person FROM PeoplePersons WHERE Person = :pid")
	res = connection.execute(s, {"pid": int(acoll)}
		).fetchall()
	#print(f"{res=}")

	if len(res) > 0:

		for (peo, per) in res:

			t = text("SELECT People, Person FROM PeoplePersons WHERE People = :ppid")
			pres = connection.execute(t, {"ppid": peo}
				).fetchall()
			#print(f"{pres=}")

			if len(pres) == 1:
				unitary = True
				break

	if not unitary:
		# Have to insert People and PeoplePerson unitary entries for collector
		s = text("INSERT INTO People(PeopleID) VALUES (:ppid)")
		connection.execute(s, {"ppid": peoplepoint})
		connection.commit()

		s = text("INSERT INTO PeoplePersons VALUES (:ppid, :perid, 1, :now)")
		connection.execute(s, {"ppid": peoplepoint, "perid": int(acoll), "now": thnow})
		connection.commit()

		peoplepoint += 1

In [ ]:
occcolls = pd.read_sql_query("SELECT OccurrenceID, CollectorVerbatim, Collector FROM Occurrences WHERE CollectorVerbatim IS NOT NULL", connection)
print(f"{occcolls.shape=}")
peoper = pd.read_sql_table("PeoplePersons", connection)
multipeople = peoper.loc[peoper.Order > 1].People.tolist()

occcolls = occcolls.merge(
	colls[["CollectorVerbatim","PersonID"]],
	how = "left",
	left_on = "CollectorVerbatim",
	right_on = "CollectorVerbatim"
).merge(
	peoper.loc[~peoper.People.isin(multipeople), ["People", "Person"]],
	how="left",
	left_on="PersonID",
	right_on="Person"
)


occcolls.shape=(42839, 3)


In [90]:
for row in occcolls.itertuples():

	s = text("UPDATE Occurrences SET Collector = :ppid WHERE OccurrenceID = :occid")
	connection.execute(s, {"ppid": row.People, "occid": row.OccurrenceID})
	connection.commit()

In [97]:
connection.close()